In [43]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


In [44]:
movies = pd.read_csv('/content/netflix_movies_detailed_up_to_2025.csv')
tv = pd.read_csv('/content/netflix_tv_shows_detailed_up_to_2025.csv')

In [45]:
netflix = pd.concat([movies, tv], ignore_index=True)

print("Total Titles:", len(netflix))


Total Titles: 32000


In [46]:
netflix = netflix[['title',
                   'genres',
                   'director',
                   'cast',
                   'description']]


In [47]:
netflix.fillna('', inplace=True)

In [48]:
netflix.drop_duplicates(subset='title', inplace=True)

print("Remaining Titles:", len(netflix))


Remaining Titles: 30639


In [49]:
netflix['tags'] = (
    netflix['genres'] + " " +
    netflix['director'] + " " +
    netflix['cast'] + " " +
    netflix['description']
)

In [50]:
new_df = netflix[['title','tags']]

new_df.head()

,title,tags
0,Shrek Forever After,"Comedy, Adventure, Fantasy, Animation, Family ..."
1,Inception,"Action, Science Fiction, Adventure Christopher..."
2,Harry Potter and the Deathly Hallows: Part 1,"Adventure, Fantasy David Yates Daniel Radcliff..."
3,Tangled,"Animation, Family, Adventure Byron Howard, Nat..."
4,How to Train Your Dragon,"Fantasy, Adventure, Animation, Family Chris Sa..."


In [51]:
tfidf = TfidfVectorizer(stop_words='english')

vectors = tfidf.fit_transform(new_df['tags'])

print(vectors.shape)

(30639, 95108)


In [52]:
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=11
)

model.fit(vectors)


NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=11)

In [53]:
def recommend(movie_name, n=10):

    movie_name = movie_name.lower().strip()

    matches = netflix[netflix['title'].str.lower() == movie_name]

    if matches.empty:
        print("Movie/Series not found!")
        return

    movie_index = matches.index[0]

    # Genres of selected title
    selected_genres = netflix.loc[movie_index, 'genres']

    # Filter titles having at least one common genre
    filtered = netflix[
        netflix['genres'].str.contains(
            '|'.join(selected_genres.split(',')),
            case=False,
            na=False
        )
    ]

    filtered = filtered.reset_index()

    # Create TF-IDF for filtered data
    tfidf = TfidfVectorizer(stop_words='english')

    vectors = tfidf.fit_transform(
        filtered['tags']
    )


    model = NearestNeighbors(metric='cosine')

    model.fit(vectors)

    idx = filtered[filtered['title'].str.lower() == movie_name].index[0]

    distances, indices = model.kneighbors(vectors[idx], n_neighbors=n+1)

    print(f"\nGenre(s): {selected_genres}\n")
    print("Top Recommendations:\n")

    for i in indices.flatten()[1:]:
        print(filtered.iloc[i]['title'])

In [54]:
def recommend(movie):

    matches = new_df[
        new_df['title'].str.contains(movie, case=False, na=False)
    ]

    if matches.empty:
        print("Movie/Series not found!")
        return

    print("Matching titles:")
    print(matches['title'].tolist())

    idx = matches.index[0]

    distances, indices = model.kneighbors(vectors[idx])

    print("\nTop Recommendations:\n")

    for i in indices.flatten()[1:]:
        print(new_df.iloc[i]['title'])

In [55]:
recommend("Avatar")


Matching titles:
['Avatar: Creating the World of Pandora', 'Capturing Avatar', 'Avatar Spirits', 'Aliens vs Avatars', "The King's Avatar: For the Glory", 'Avatar: The Way of Water', 'Avatar: The Deep Dive - A Special Edition of 20/20', 'Avataro Sentai Donbrothers vs. Zenkaiger', 'Avatar: Fire and Ash', "The King's Avatar", 'Avataro Sentai Donbrothers', 'Avatar the Last Airbender']

Top Recommendations:

Titanic: 20 Years Later with James Cameron
Avatar: The Deep Dive - A Special Edition of 20/20
Capturing Avatar
James Cameron's Story of Science Fiction
Wissen vor acht – Natur
Shrek: Once Upon a Time
The Legend of Korra
Side by Side
Straw Dogs
Jumping the Broom


In [56]:
recommend("wednesday")

Matching titles:
['Adult Wednesday Addams', 'Wednesday Downtown', 'Wednesday 3:30 PM', 'Wednesday']

Top Recommendations:

Vacation of Love
Impression of Youth
May December Love
Sisterhood
성시경의 먹을텐데
If By Life You Were Deceived
The Fairy Fox
The Legend of Xiao Zhuang
Road to Rebirth
掩护


In [57]:
netflix[netflix['title'].str.contains("Avatar", case=False, na=False)][['title']]

,title
150,Avatar: Creating the World of Pandora
229,Capturing Avatar
764,Avatar Spirits
1645,Aliens vs Avatars
9828,The King's Avatar: For the Glory
12006,Avatar: The Way of Water
12250,Avatar: The Deep Dive - A Special Edition of 2...
13807,Avataro Sentai Donbrothers vs. Zenkaiger
15078,Avatar: Fire and Ash
23401,The King's Avatar


In [58]:
import ipywidgets as widgets
from IPython.display import display, clear_output

movie_input = widgets.Text(
    value='',
    placeholder='Enter movie or TV show name',
    description='Title:',
    layout=widgets.Layout(width='400px')
)

button = widgets.Button(
    description='Recommend',
    button_style='success'
)

output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()

        movie = movie_input.value.strip()

        if movie == "":
            print("Please enter a movie or TV show name.")
            return

        recommend(movie)

button.on_click(on_button_clicked)

display(movie_input)
display(button)
display(output)


Text(value='', description='Title:', layout=Layout(width='400px'), placeholder='Enter movie or TV show name')

Button(button_style='success', description='Recommend', style=ButtonStyle())

Output()